# 🏠 California Housing Price — Linear Regression Workshop
**AI Club — Hands-On Session**

This notebook is the live coding companion for the *Linear Regression* workshop.
We walk through every step: data cleaning → feature engineering → model fitting → evaluation → diagnostics.

---
> **Dataset:** [California Housing Prices](https://www.kaggle.com/datasets/camnugent/california-housing-prices) — 1990 US Census  
> **Goal:** Predict the **median house value** of a block group from 9 features.


## 0 · Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import warnings; warnings.filterwarnings("ignore")

# ── Plot style ────────────────────────────────────────────────────────────────
BG, CARD, ACC, RED, GOLD = "#0F172A", "#1E293B", "#38BDF8", "#F87171", "#FBBF24"
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": CARD,
    "axes.edgecolor": "#94A3B8", "axes.labelcolor": "#94A3B8",
    "xtick.color": "#94A3B8", "ytick.color": "#94A3B8",
    "text.color": "#E2E8F0", "grid.color": "#2D3D50",
    "grid.linewidth": 0.5, "font.family": "sans-serif",
})

def save_fig(name):
    plt.savefig(f"figures/{name}.png", dpi=150, bbox_inches="tight",
                facecolor=BG, edgecolor="none")
    print(f"  ✓ saved figures/{name}.png")


## 1 · Load & Inspect the Data

We immediately split off a **hold-out test set** that we won't look at until final evaluation.


In [ ]:
housing = pd.read_csv("housing.csv")

# Hold-out split — do this BEFORE any exploration
housing, test_set = train_test_split(housing, test_size=0.2, random_state=4)

print(f"Training set : {housing.shape}")
print(f"Test set     : {test_set.shape}")
housing.head()


In [ ]:
housing.info()

In [ ]:
housing.describe().round(2)

## 2 · Missing Values

`total_bedrooms` has 207 nulls (~2.5 %). We inspect the distribution to choose
mean vs. median imputation — a skewed variable calls for the **median**.


In [ ]:
housing.isna().sum()[housing.isna().sum() > 0]


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(housing["total_bedrooms"].dropna(), bins=40, kde=True,
             ax=ax, color=ACC, edgecolor=CARD)
ax.set_title("Distribution of total_bedrooms (right-skewed → use median)")
ax.set_xlabel("total_bedrooms"); ax.grid(True, alpha=0.4)
plt.tight_layout(); save_fig("total_bedrooms_dist"); plt.show()


In [ ]:
housing["total_bedrooms"].fillna(housing["total_bedrooms"].median(), inplace=True)
print("Remaining nulls:", housing.isna().sum().sum())


## 3 · Categorical Encoding

`ocean_proximity` has 5 categories. We use **dummy encoding** (drop_first=True)
to avoid the multicollinearity trap in linear models.


In [ ]:
print(housing["ocean_proximity"].value_counts())


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
housing["ocean_proximity"].value_counts().plot(kind="bar", ax=ax, color=ACC,
                                               edgecolor=CARD, width=0.6)
ax.set_title("ocean_proximity — value counts")
ax.set_xlabel(""); ax.tick_params(axis="x", rotation=30)
ax.grid(True, axis="y", alpha=0.4)
plt.tight_layout(); plt.show()


In [ ]:
housing = pd.get_dummies(housing, columns=["ocean_proximity"],
                         prefix="ocean", drop_first=True)
# Cast to int for cleanliness
ocean_cols = [c for c in housing.columns if c.startswith("ocean")]
housing[ocean_cols] = housing[ocean_cols].astype(int)
housing.head(3)


## 4 · Outliers & Target Transformation

The target `median_house_value` is capped at **$500 k** (census artefact).
A **log transform** compresses the scale and makes the distribution closer to normal —
a key assumption for linear regression.


In [ ]:
num_cols = ["housing_median_age","total_rooms","total_bedrooms",
            "population","households","median_income","median_house_value"]

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, col in zip(axes.flat, num_cols):
    sns.boxplot(x=housing[col], ax=ax, color=ACC, flierprops={"marker":"o","markersize":2})
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("")
axes.flat[-1].set_visible(False)
fig.suptitle("Outlier Overview — Numerical Features", fontweight="bold", y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.histplot(housing["median_house_value"], bins=50, kde=True,
             ax=axes[0], color=RED, edgecolor=CARD)
axes[0].set_title("Before log transform (clipped at $500 k)")

housing["median_house_value"] = np.log(housing["median_house_value"])

sns.histplot(housing["median_house_value"], bins=50, kde=True,
             ax=axes[1], color=ACC, edgecolor=CARD)
axes[1].set_title("After log transform (approx. normal)")

for ax in axes: ax.grid(True, alpha=0.4)
plt.tight_layout(); save_fig("target_log_transform"); plt.show()


## 5 · Feature Engineering

Raw counts (rooms, bedrooms, population) are correlated with each other and with
`households`. Dividing them by households gives **per-household ratios** that are
more meaningful and reduce multicollinearity.


In [ ]:
housing["rooms_per_household"]      = housing["total_rooms"]      / housing["households"]
housing["bedrooms_per_room"]        = housing["total_bedrooms"]   / housing["total_rooms"]
housing["population_per_household"] = housing["population"]       / housing["households"]

housing.drop(["total_rooms","total_bedrooms","population","households"],
             axis=1, inplace=True)


In [ ]:
eng_features = ["median_income","housing_median_age","rooms_per_household",
                "bedrooms_per_room","population_per_household","median_house_value"]

fig, ax = plt.subplots(figsize=(8, 6))
corr = housing[eng_features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            ax=ax, linewidths=0.4, linecolor=BG, cbar_kws={"shrink":0.8})
ax.set_title("Correlation Matrix — Engineered Features", fontweight="bold")
ax.tick_params(colors="#E2E8F0", labelsize=9)
plt.tight_layout(); save_fig("fig_bonus_correlation_heatmap"); plt.show()
print("\nCorrelation with target (median_house_value):")
print(corr["median_house_value"].sort_values(key=abs, ascending=False).round(3))


## 6 · Feature Scaling & Train / Val Split

`StandardScaler` (zero mean, unit variance) ensures no single feature dominates
the OLS solution because of its numeric scale.


In [ ]:
X = housing.drop("median_house_value", axis=1)
y = housing["median_house_value"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.125, random_state=4)

scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_val_s   = pd.DataFrame(scaler.transform(X_val),   columns=X_val.columns)

print(f"Training   : {X_train_s.shape}")
print(f"Validation : {X_val_s.shape}")


## 7 · Evaluation Helpers

Three diagnostic plots — these are the standard visual checks for any regression model.


In [ ]:
def evaluate(name, y_true, y_pred):
    r2  = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    print(f"{'─'*40}")
    print(f"  Model : {name}")
    print(f"  R²    : {r2:.4f}")
    print(f"  RMSE  : {rmse:.4f}")
    print(f"  MSE   : {mse:.4f}")
    return {"model": name, "R2": r2, "RMSE": rmse, "MSE": mse}

def plot_diagnostics(y_true, y_pred, name, save=False):
    resid = y_true - y_pred
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # Actual vs predicted
    axes[0].scatter(y_true, y_pred, color=ACC, alpha=0.4, s=12, rasterized=True)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    axes[0].plot(lims, lims, color=RED, lw=2, ls="--", label="y = ŷ")
    axes[0].set_xlabel("Actual"); axes[0].set_ylabel("Predicted")
    axes[0].set_title("Actual vs. Predicted"); axes[0].legend(); axes[0].grid(True, alpha=0.4)

    # Residual plot
    axes[1].scatter(y_pred, resid, color=ACC, alpha=0.4, s=12, rasterized=True)
    axes[1].axhline(0, color=RED, lw=2, ls="--")
    axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residuals (y − ŷ)")
    axes[1].set_title("Residual Plot"); axes[1].grid(True, alpha=0.4)

    # Residual distribution
    sns.histplot(resid, bins=40, kde=True, ax=axes[2],
                 color=ACC, edgecolor=CARD, line_kws={"color": GOLD, "lw": 2})
    axes[2].axvline(0, color=RED, lw=2, ls="--")
    axes[2].set_xlabel("Residual"); axes[2].set_title("Residual Distribution")
    axes[2].grid(True, alpha=0.4)

    fig.suptitle(f"Regression Diagnostics — {name}", fontweight="bold", y=1.01)
    plt.tight_layout()
    if save:
        save_fig(f"diagnostics_{name.lower().replace(' ','_')}")
    plt.show()

results = []


## 8 · Linear Regression

The baseline model. We fit it on scaled training data, then evaluate on the validation set.


In [ ]:
lr = LinearRegression()
lr.fit(X_train_s, y_train)

y_val_pred = lr.predict(X_val_s)
results.append(evaluate("Linear Regression", y_val, y_val_pred))
plot_diagnostics(y_val, y_val_pred, "Linear Regression", save=True)


In [ ]:
# ── Coefficient inspection ────────────────────────────────────────────────────
coef_df = (pd.DataFrame({"Feature": X_train_s.columns, "Coefficient": lr.coef_})
             .sort_values("Coefficient", key=abs, ascending=True))

fig, ax = plt.subplots(figsize=(7, 5))
colors = [ACC if c > 0 else RED for c in coef_df["Coefficient"]]
ax.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors, edgecolor=CARD)
ax.axvline(0, color="#94A3B8", lw=1, ls="--")
ax.set_title("Feature Coefficients — Linear Regression", fontweight="bold")
ax.set_xlabel("Coefficient value")
ax.grid(True, axis="x", alpha=0.4)
plt.tight_layout(); plt.show()


## 9 · Polynomial Regression (degree = 2)

By expanding features to degree 2 we let the model capture non-linear patterns —
while the model is still *linear in its coefficients*.


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

poly_lr = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    StandardScaler(),
    LinearRegression()
)
poly_lr.fit(X_train_s, y_train)

y_val_pred_poly = poly_lr.predict(X_val_s)
results.append(evaluate("Polynomial Regression (deg 2)", y_val, y_val_pred_poly))
plot_diagnostics(y_val, y_val_pred_poly, "Polynomial Regression")


## 10 · Ridge Regression (L2 Regularization)

Ridge adds a penalty on large coefficients:  
`J(θ) = MSE(θ) + λ · Σ θᵢ²`

We use cross-validated `RidgeCV` to find the best `λ` automatically.


In [ ]:
from sklearn.linear_model import RidgeCV

ridge = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    StandardScaler(),
    RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10, 100], cv=5,
            scoring="neg_mean_squared_error")
)
ridge.fit(X_train_s, y_train)

best_alpha = ridge.named_steps["ridgecv"].alpha_
print(f"Best α (lambda): {best_alpha}")

results.append(evaluate("Ridge (deg 2)", y_val, ridge.predict(X_val_s)))
plot_diagnostics(y_val, ridge.predict(X_val_s), "Ridge Regression")


## 11 · Lasso Regression (L1 Regularization)

Lasso adds an L1 penalty that can **shrink some coefficients to exactly zero** —
effectively performing automatic feature selection.


In [ ]:
from sklearn.linear_model import LassoCV

lasso = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    StandardScaler(),
    LassoCV(alphas=[0.001, 0.01, 0.1, 1, 10], cv=5)
)
lasso.fit(X_train_s, y_train)

best_alpha_l = lasso.named_steps["lassocv"].alpha_
print(f"Best α (lambda): {best_alpha_l}")

results.append(evaluate("Lasso (deg 2)", y_val, lasso.predict(X_val_s)))
plot_diagnostics(y_val, lasso.predict(X_val_s), "Lasso Regression")


## 12 · Model Comparison

A side-by-side summary of all linear models on the validation set.


In [ ]:
results_df = pd.DataFrame(results).set_index("model").sort_values("R2", ascending=False)
display(results_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].barh(results_df.index, results_df["R2"], color=ACC, edgecolor=CARD)
axes[0].set_xlabel("R²"); axes[0].set_title("R² (higher = better)", fontweight="bold")
axes[0].axvline(results_df["R2"].max(), color=GOLD, lw=1.5, ls="--")
axes[0].grid(True, axis="x", alpha=0.4)

axes[1].barh(results_df.index, results_df["RMSE"], color=RED, edgecolor=CARD)
axes[1].set_xlabel("RMSE"); axes[1].set_title("RMSE (lower = better)", fontweight="bold")
axes[1].axvline(results_df["RMSE"].min(), color=GOLD, lw=1.5, ls="--")
axes[1].grid(True, axis="x", alpha=0.4)

plt.tight_layout(); save_fig("model_comparison"); plt.show()


## 13 · Key Takeaways

| Finding | Implication |
|---------|-------------|
| All linear models show heteroscedasticity | Variance of errors grows with prediction → log-transform helped, but residuals still fan out |
| Polynomial degree-2 outperforms plain OLS | Non-linear relationships exist in the data |
| Ridge ≈ Lasso performance | Regularization stabilises but doesn't dramatically improve on this dataset |
| `median_income` dominates coefficients | Most predictive single feature |

> **Next steps to explore:**  
> 🌳 Decision Tree / Random Forest — handle non-linearity natively  
> ⚡ XGBoost — sequential boosting, state-of-the-art for tabular data  
> 🔬 Tune polynomial degree and regularization strength  
